# Banking77 integrated application demonstration

This notebook exercises the application contract and simulated tool using a deterministic mock generator. It does not load the LLM and does not produce model evaluation results. Replace the mock generator with `generate_response(model, tokenizer, prompt, decoding_config)` for a real run.

In [ ]:
from pathlib import Path
import json, sys
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.pipeline import run_pipeline
from src.output.schema import parse_output
from src.evaluation.metrics import evaluate_records

## Structured output and simulated tool flow

In [ ]:
mock_action = {
    'intent': 'lost_or_stolen_card',
    'response': 'I can help with the lost card request.',
    'action_required': True,
    'action': {'tool': 'simulate_card_lock', 'arguments': {'card_ref': 'demo-card-001', 'reason': 'lost'}},
}
result = run_pipeline('I lost my card', lambda prompt: json.dumps(mock_action))
result

## Failure handling examples

In [ ]:
invalid_json = run_pipeline('Where is my card?', lambda prompt: 'not valid JSON')
blocked = run_pipeline('My PIN is 1234', lambda prompt: (_ for _ in ()).throw(AssertionError('must not generate')))
invalid_json, blocked

## Scoring contract example
Use actual raw model outputs and the fixed evaluation sample to report metrics. The toy record below only demonstrates metric field names.

In [ ]:
toy = [{'reference_intent': 'card_arrival', 'raw_output': json.dumps({'intent':'card_arrival','response':'Check the official app.','action_required':False,'action':None})}]
evaluate_records(toy)